# PUMA-MV partial-annotation benchmark — Kaggle 2 GPU

`validation_*` uses 0–80% train / 80–90% validation. Only after freezing threshold and NMS should `final_*` retrain on 0–90% and evaluate 90–100%. Partial annotations and evaluator GT are generated under `/kaggle/working`; the read-only dataset is never modified.

In [ ]:
# ===== User configuration =====
REPO_URL = 'https://github.com/x23d8/MVDet.git'
BRANCH = 'puma'
DATASET_ROOT = '/kaggle/input/datasets/lee735/thesis-dataset-new/Wildtrack/Wildtrack'  # leave empty to auto-discover
MODE = 'validation_scar'  # smoke | validation_scar | validation_sar | final_scar | final_sar | baselines
EPOCHS = 20
DEVICES = '0,1'
BATCH_SIZE = 2             # global DataParallel batch: one sample/GPU
NUM_WORKERS = 4
FEATURE_CHANNELS = 32
FUSED_CHANNELS = 64
QUERY_CHANNELS = 64
PARALLEL_VIEW_ENCODING = False  # safer on T4; True is faster but uses a large cuDNN workspace
DROP_RATIO = 60
DROP_RATIOS = '0 20 45 60'  # final_scar/baselines; final_sar skips 0
SEEDS = '1 2 3'
CLS_THRES = 0.4            # replace with validation-selected value for final_*
NMS_RADIUS_M = 0.3         # replace with validation-selected value for final_*
RUN_TESTS = True
RUN_SMOKE = True
RESUME_SOURCE = ''         # optional attached run directory containing training_state.pth


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, torch

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PROJECT = WORK / 'puma_mv_project'
SPLIT_WORK = WORK / 'puma_splits'
GT_PATH = WORK / 'generated_gt' / 'wildtrack_gt.txt'

def run(command, cwd=None, env=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, PROJECT])
MVDET = PROJECT / 'MVDetBRL'
required = [MVDET / 'main.py', MVDET / 'multiview_detector/models/puma_hybrid_detector.py', MVDET / 'run_puma_validation.sh']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Configured branch does not contain PUMA-MV files: ' + ', '.join(missing))
run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT)

def is_wildtrack_root(path):
    return (path / 'annotations_positions').is_dir() and (path / 'Image_subsets' / 'C1').is_dir() and (path / 'calibrations').is_dir()

configured_root = Path(DATASET_ROOT) if DATASET_ROOT.strip() else None
if configured_root is not None and is_wildtrack_root(configured_root):
    WILDTRACK = configured_root
    print('Using configured DATASET_ROOT')
else:
    if configured_root is not None:
        print(f'Configured DATASET_ROOT is unavailable or incomplete: {configured_root}; auto-discovering below {INPUT}')
    roots = []
    for annotations in INPUT.rglob('annotations_positions'):
        root = annotations.parent
        if is_wildtrack_root(root):
            roots.append(root)
    if not roots:
        raise FileNotFoundError('No Wildtrack root with annotations_positions, Image_subsets/C1, and calibrations found')
    WILDTRACK = sorted(set(roots), key=lambda path: (len(str(path)), str(path)))[0]
    print('Auto-discovered Wildtrack dataset')
camera_counts = {f'C{i}': len(list((WILDTRACK / 'Image_subsets' / f'C{i}').glob('*.png'))) for i in range(1, 8)}
annotation_count = len(list((WILDTRACK / 'annotations_positions').glob('*.json')))
print('Wildtrack:', WILDTRACK)
print('Files:', camera_counts, 'annotations=', annotation_count)
if min(camera_counts.values()) < 400 or annotation_count < 400:
    raise RuntimeError('Wildtrack is incomplete; expected at least 400 frames per camera and 400 annotation files')
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('CUDA:', torch.cuda.is_available(), gpu_names)
if MODE != 'smoke' and len([d for d in DEVICES.split(',') if d.strip()]) == 2 and torch.cuda.device_count() < 2:
    raise RuntimeError('Select two GPUs in Kaggle Settings; this run requests devices 0,1')

try:
    import kornia  # noqa: F401
except ImportError:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'kornia==0.7.3'])
if RUN_TESTS:
    run([sys.executable, '-m', 'pytest', 'tests', '-q'], cwd=MVDET)


In [ ]:
# Build deterministic partial annotations in writable storage.
mechanisms = []
if MODE in {'validation_scar', 'final_scar', 'baselines'}:
    mechanisms.append('scar')
if MODE in {'validation_sar', 'final_sar'}:
    mechanisms.append('visibility_sar')
grid_drops = [int(value) for value in DROP_RATIOS.split()]
settings = [f'drop{value}' for value in grid_drops if value > 0] if MODE in {'final_scar', 'final_sar', 'baselines'} else [f'drop{DROP_RATIO}']
for mechanism in mechanisms:
    run([
        sys.executable, PROJECT / 'simulate_dropped_annotations.py',
        '-d', 'Wildtrack', '-r', WILDTRACK, '-s', *settings,
        '--seed', '1', '--mechanism', mechanism, '--output-root', SPLIT_WORK,
    ])

resume_dir = ''
if RESUME_SOURCE:
    source = Path(RESUME_SOURCE)
    candidates = [source] if (source / 'training_state.pth').is_file() else [p.parent for p in source.rglob('training_state.pth')]
    if len(candidates) != 1:
        raise RuntimeError(f'RESUME_SOURCE must resolve to exactly one run, found {len(candidates)}')
    destination = WORK / 'resume_run'
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(candidates[0], destination)
    resume_dir = str(destination)

env = os.environ.copy()
env.update({
    'DATASET': 'wildtrack', 'DATA_PATH': str(WILDTRACK),
    'SPLIT_ROOT': str(SPLIT_WORK / 'drop_annotations'), 'GT_PATH': str(GT_PATH),
    'DEVICES': DEVICES, 'BATCH_SIZE': str(BATCH_SIZE),
    'NUM_WORKERS': str(NUM_WORKERS), 'EPOCHS': str(EPOCHS),
    'FEATURE_CHANNELS': str(FEATURE_CHANNELS), 'FUSED_CHANNELS': str(FUSED_CHANNELS),
    'QUERY_CHANNELS': str(QUERY_CHANNELS),
    'PARALLEL_VIEW_ENCODING': '1' if PARALLEL_VIEW_ENCODING else '0',
    'DROP_RATIO': str(DROP_RATIO), 'DROP_RATIOS': DROP_RATIOS,
    'SEEDS': SEEDS, 'CLS_THRES': str(CLS_THRES), 'NMS_RADIUS_M': str(NMS_RADIUS_M),
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
if resume_dir:
    env['RESUME_DIR'] = resume_dir
if MODE == 'validation_sar':
    env['PROPENSITY_MODE'] = 'sar'
    env['TRAIN_ANNOTATION_DIR'] = str(SPLIT_WORK / 'drop_annotations' / 'visibility_sar' / f'drop_{DROP_RATIO}' / 'annotations_positions')
if MODE == 'final_sar':
    env['DROP_RATIOS'] = ' '.join(str(value) for value in grid_drops if value > 0)

if RUN_SMOKE:
    smoke_command = [
        sys.executable, 'tools/smoke_real_wildtrack.py', WILDTRACK,
        '--gt-path', GT_PATH, '--consistency-pass',
        '--feature-channels', FEATURE_CHANNELS, '--fused-channels', FUSED_CHANNELS,
        '--query-channels', QUERY_CHANNELS,
    ]
    if PARALLEL_VIEW_ENCODING:
        smoke_command.append('--parallel-view-encoding')
    run(smoke_command, cwd=MVDET, env=env)


In [ ]:
# Train/evaluate. Results are archived even when a run fails.
(MVDET / 'logs').mkdir(exist_ok=True)
try:
    if MODE == 'smoke':
        pass
    elif MODE in {'validation_scar', 'validation_sar'}:
        run(['bash', 'run_puma_validation.sh'], cwd=MVDET, env=env)
    elif MODE == 'final_scar':
        run(['bash', 'run_puma_grid.sh'], cwd=MVDET, env=env)
    elif MODE == 'final_sar':
        run(['bash', 'run_puma_sar_grid.sh'], cwd=MVDET, env=env)
    elif MODE == 'baselines':
        run(['bash', 'run_required_baselines.sh'], cwd=MVDET, env=env)
    else:
        raise ValueError(f'Unknown MODE={MODE!r}')
finally:
    if resume_dir and Path(resume_dir).is_dir():
        shutil.copytree(resume_dir, MVDET / 'logs' / 'resumed_run', dirs_exist_ok=True)
    archive = shutil.make_archive(str(WORK / 'puma_mv_results'), 'zip', MVDET / 'logs')
    print('Results archive:', archive)
